# Stage 11 — Final reviewer sensitivity run

One-click execution for the two remaining inference-only reviewer analyses:
1. `best.pt` versus primary `last.pt` checkpoint sensitivity (Stage 11D).
2. image-cluster bootstrap 95% CI for mAP50:95 domain gaps using primary `last.pt` (Stage 11E2).

No new training, fine-tuning, adaptation, or threshold tuning is performed.

Analysis code is pinned to immutable Git commit `10e7b7e116288869647c61287d0c88056bf0a972`.

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil, subprocess, sys, hashlib, zipfile
drive.mount('/content/drive',force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
RUN_ROOT=DRIVE/'Stage5_Baselines'
STAGE6=DRIVE/'Stage6_Heldout_Evaluation'
OUT_D=DRIVE/'Stage11D_Best_Checkpoint_Sensitivity'
OUT_E=DRIVE/'Stage11E2_MAP_Bootstrap'
OUT_D.mkdir(parents=True,exist_ok=True); OUT_E.mkdir(parents=True,exist_ok=True)
if not shutil.which('nvidia-smi'): raise RuntimeError('Select T4 GPU before Run all.')
subprocess.run(['nvidia-smi'],check=True)
print('GPU: PASS')

In [ ]:
PARTS=[DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_00',DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_01',DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_02']
BUNDLE=Path('/content/Stage5_Frozen_h9_v2_Data_Bundle.zip')
EXPECTED='dd8ce20f2650848133ce4323b00139684b0d4ac8b7db618e8df469244221cee9'
h=hashlib.sha256()
with BUNDLE.open('wb') as out:
    for p in PARTS:
        if not p.is_file(): raise FileNotFoundError(p)
        with p.open('rb') as f:
            for ch in iter(lambda:f.read(8*1024*1024),b''):
                out.write(ch); h.update(ch)
assert h.hexdigest()==EXPECTED
DATA=Path('/content/stage11_final_data')
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir()
with zipfile.ZipFile(BUNDLE) as z: z.extractall(DATA)
Y_DATA=DATA/'h9_v2/Y-PPE-h9-v2/data.yaml'
C_DATA=DATA/'h9_v2/Construction-PPE-h9-v2/data.yaml'
print('FROZEN DATA: PASS')

In [ ]:
REPO=Path('/content/Y-PPE-Detection-RFDETR')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','https://github.com/EZZALDEEN049/Y-PPE-Detection-RFDETR.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','10e7b7e116288869647c61287d0c88056bf0a972'],check=True)
HEAD=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert HEAD=='10e7b7e116288869647c61287d0c88056bf0a972', HEAD
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.156','PyYAML==6.0.3','pandas'],check=True)
import ultralytics, torch
assert ultralytics.__version__=='8.4.156'
assert torch.cuda.is_available()
print('FROZEN ANALYSIS CODE SHA:',HEAD)
print('CODE ENV: PASS')

In [ ]:
print('START STAGE 11D')
SD=REPO/'cross_domain_reliability/stage11_major_revision/run_stage11d_best_checkpoint_sensitivity.py'
subprocess.run([sys.executable,str(SD),'--repo',str(REPO),'--stage5-root',str(RUN_ROOT),'--y-data',str(Y_DATA),'--c-data',str(C_DATA),'--out',str(OUT_D),'--original-group-summary',str(STAGE6/'stage6_group_summary.csv'),'--device','0','--batch','8','--imgsz','640','--ap-conf','0.001','--operating-conf','0.25','--nms-iou','0.70','--match-iou','0.50'],check=True)
print('STAGE 11D: PASS')

In [ ]:
print('START STAGE 11E2')
SE=REPO/'cross_domain_reliability/stage11_major_revision/run_stage11e2_map_bootstrap.py'
subprocess.run([sys.executable,str(SE),'--stage5-root',str(RUN_ROOT),'--y-data',str(Y_DATA),'--c-data',str(C_DATA),'--out',str(OUT_E),'--device','0','--batch','8','--imgsz','640','--ap-conf','0.001','--nms-iou','0.70','--bootstrap-reps','2000','--bootstrap-seed','20261009'],check=True)
print('STAGE 11E2: PASS')

In [ ]:
import pandas as pd
from IPython.display import display
print('BEST.PT VS LAST.PT')
display(pd.read_csv(OUT_D/'stage11d_best_vs_last_comparison.csv'))
print('mAP IMAGE-CLUSTER BOOTSTRAP')
display(pd.read_csv(OUT_E/'stage11e2_map_bootstrap_summary.csv'))
print('STAGE11_FINAL_REVIEWER_SENSITIVITY_COMPLETE')